# Virchow qualitative error review (C1–C4)

Run **top-to-bottom**. Edit **CONFIG** if your paths differ.

- Exports PNGs + `selected_cases.csv` + `review_labels_template.csv` + `gallery.html` under `reports/qualitative_error_analysis/`.
- Buckets and sampling match `docs/qualitative_error_analysis_protocol.md` (20 per bucket, seed 42) + methodology Ch.6 (entropy threshold = 90th pct of H on that split).
- **Checklist definitions:** `docs/qualitative_error_analysis_protocol.md` §6. Fill `review_labels_template.csv` (e.g. in Excel); save as `review_labels_completed.csv` when done.

**Requires:** `test_predictions.npz` in each eval folder (re-run Virchow eval without `--no-save-test-preds` if missing).

## 0 — Optional: install (run once per environment)

In [ ]:
%pip install -q h5py pillow numpy

## 1 — Repo root + import export helper

In [ ]:
from pathlib import Path
import sys

_cwd = Path.cwd().resolve()
if (_cwd / "scripts" / "export_qualitative_review_patches.py").is_file():
    REPO = _cwd
elif (_cwd.parent / "scripts" / "export_qualitative_review_patches.py").is_file():
    REPO = _cwd.parent
else:
    REPO = Path(r"C:\GP_ECG")  # <-- edit if needed

sys.path.insert(0, str(REPO / "scripts"))
from export_qualitative_review_patches import export_qualitative_review

print("REPO =", REPO)

## 2 — CONFIG (edit paths)

- `EVAL_ROOT`: folder that contains the four eval subfolders from Colab.
- H5 paths must be the **same** `test_x.h5` files used when `test_predictions.npz` was produced (same row order as NPZ).

In [ ]:
EVAL_ROOT = REPO / "experiments" / "virchow_colab" / "evals_cross_domain"

PCAM_PRE_TEST_X = REPO / "pcam_data" / "preprocessed_macenko_benchmark_style" / "test_x.h5"
PCAM_RAW_TEST_X = REPO / "pcam_data" / "test" / "camelyonpatch_level_2_split_test_x.h5"

CAM17_PRE_TEST_X = (
    REPO / "data" / "wilds" / "camelyon17_h5_full_oodval" / "preprocessed_macenko_benchmark_style" / "test_x.h5"
)
CAM17_RAW_TEST_X = REPO / "data" / "wilds" / "camelyon17_h5_full_oodval" / "test_x.h5"

OUT_ROOT = REPO / "reports" / "qualitative_error_analysis" / "virchow_c1_c4"

# Extra bucket: confident errors (methodology §6). Set False for protocol-only four buckets.
INCLUDE_CONFIDENT_ERRORS = True

CONDITIONS = [
    {
        "id": "C1",
        "eval_subdir": "pcam_trained_on_pcam_test",
        "direction": "PCam_test (in-domain)",
        "pre_x": PCAM_PRE_TEST_X,
        "raw_x": PCAM_RAW_TEST_X,
    },
    {
        "id": "C2",
        "eval_subdir": "pcam_trained_on_cam17_test",
        "direction": "PCam_train -> CAMELYON17_test",
        "pre_x": CAM17_PRE_TEST_X,
        "raw_x": CAM17_RAW_TEST_X,
    },
    {
        "id": "C3",
        "eval_subdir": "cam17_trained_on_cam17_test",
        "direction": "CAMELYON17_test (in-domain)",
        "pre_x": CAM17_PRE_TEST_X,
        "raw_x": CAM17_RAW_TEST_X,
    },
    {
        "id": "C4",
        "eval_subdir": "cam17_trained_on_pcam_test",
        "direction": "CAMELYON17_train -> PCam_test",
        "pre_x": PCAM_PRE_TEST_X,
        "raw_x": PCAM_RAW_TEST_X,
    },
]

for c in CONDITIONS:
    npz = EVAL_ROOT / c["eval_subdir"] / "test_predictions.npz"
    print(c["id"], "NPZ exists:", npz.is_file(), "|", npz)

## 3 — Export all four conditions (PNG + CSV + gallery)

Re-run this cell if you change CONFIG or want to refresh samples.

In [ ]:
OUT_DIRS = {}
for c in CONDITIONS:
    npz = EVAL_ROOT / c["eval_subdir"] / "test_predictions.npz"
    if not npz.is_file():
        raise FileNotFoundError(f"Missing {npz} — need test_predictions.npz for {c['id']}")
    if not c["pre_x"].is_file():
        raise FileNotFoundError(f"Missing preprocessed test_x: {c['pre_x']}")
    raw = c["raw_x"] if c["raw_x"].is_file() else None
    if raw is None:
        print(f"[warn] {c['id']}: raw test_x not found, preprocessed PNGs only:", c["raw_x"])
    od = OUT_ROOT / f"{c['id']}_{c['eval_subdir']}"
    export_qualitative_review(
        npz,
        c["pre_x"],
        od,
        raw_test_x=raw,
        condition_id=c["id"],
        direction=c["direction"],
        include_confident_errors=INCLUDE_CONFIDENT_ERRORS,
    )
    OUT_DIRS[c["id"]] = od

print("Done. Output folders:")
for k, v in OUT_DIRS.items():
    print(f"  {k}: {v}")

## 4 — Checklist (per case)

In each `.../review_labels_template.csv`, use **Present**, **Absent**, or **Unclear** for:

1. tissue scarcity  
2. artifact burden  
3. borderline morphology  
4. small-focus lesion pattern  
5. color/stain atypia  
6. patch-context limitation  
7. free-text note (1–2 lines)

Follow **`review_order`** column (randomized). Save completed file as **`review_labels_completed.csv`** in the same folder.

## 5 — Helper: show patches in the notebook (optional preview)

Full set: open `gallery.html` in that folder in a browser.

In [ ]:
import csv
from IPython.display import display, Markdown, Image


def preview_condition(out_dir: Path, max_per_bucket: int = 6):
    """Show up to max_per_bucket preprocessed PNGs per bucket (raw omitted here; use gallery.html for pairs)."""
    out_dir = Path(out_dir)
    sc = out_dir / "selected_cases.csv"
    if not sc.is_file():
        print("No selected_cases.csv — run section 3 first.")
        return
    rows = []
    with open(sc, newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            rows.append(r)
    by_bucket = {}
    for r in rows:
        by_bucket.setdefault(r["bucket"], []).append(r)
    display(Markdown(f"### {out_dir.name}"))
    display(Markdown(f"**gallery:** `{out_dir / 'gallery.html'}`  \n**checklist:** `{out_dir / 'review_labels_template.csv'}`"))
    for b in sorted(by_bucket.keys()):
        display(Markdown(f"#### Bucket `{b}`"))
        for r in by_bucket[b][:max_per_bucket]:
            pth = Path(r["figure_preprocessed"])
            if pth.is_file():
                cap = f"{r['case_id']} idx={r['h5_index']} y={r['y_true']} pred={r['y_hat']} p={float(r['p_cal']):.3f}"
                display(Markdown(cap))
                display(Image(filename=str(pth), width=220))


# Uncomment to preview all four in one go (many images):
# for cid in ("C1", "C2", "C3", "C4"):
#     preview_condition(OUT_DIRS[cid])

### C1 — inspect + fill checklist

In [ ]:
preview_condition(OUT_DIRS["C1"])

### C2 — inspect + fill checklist

In [ ]:
preview_condition(OUT_DIRS["C2"])

### C3 — inspect + fill checklist

In [ ]:
preview_condition(OUT_DIRS["C3"])

### C4 — inspect + fill checklist

In [ ]:
preview_condition(OUT_DIRS["C4"])